In [1]:
from environment import Environment
from nnsight import LanguageModel
import sys
import torch as t
from getpass import getpass
import os

REPLICATE_API_TOKEN = getpass()
os.environ["REPLICATE_API_TOKEN"] = "r8_01YrzTehxtHci1pTUY0jbBit87BcqDI4Ubi3l"

In [ ]:
model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [ ]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

local_dir = "../jbloom_dictionaries"

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir=local_dir)

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

In [ ]:
env = Environment(
    model=model, 
    sae_list=sae_list,
)

In [ ]:
env.load(
    layer=10,
    feature_id=7000,
)

In [ ]:
long_explanation_list = env.explainer.explain()

env.state.history.append({"role":"system","message":long_explanation_list})

In [ ]:
from condenser import condense
from prompting import get_simple_condenser_template

explanation_list, output = condense(long_explanation_list, return_output=True)

env.state.history.append({"role":"section","message":"Running condenser."})
env.state.history.append({"role":"user","message":get_simple_condenser_template(long_explanation_list)})
env.state.history.append({"role":"assistant","message":"".join(output)})

In [ ]:
d_scores_list = env.d_scorer.score(explanation_list)
pruned_explanation_list = [explanation_list[i] for i in range(len(explanation_list)) if d_scores_list[i][0] > 0.6 and d_scores_list[i][1] < 0.3]

pruned_explanation_list

In [ ]:
g_scores_list = env.gen_scorer.score(explanation_list, sae_list[layer])

In [ ]:
import importlib
import utils
importlib.reload(utils)

utils.log_conversation(env.state.history, "conversation.txt")

In [ ]:
for i in range(len(explanation_list)):
  print(f"Explanation: {explanation_list[i]}")
  print(f"Detection rate: {d_scores_list[i][0]:.2f}")
  print(f"False positive rate: {d_scores_list[i][1]:.2f}")
  print(f"Generation score: {g_scores_list[i]:.2f}")
  print()